# 04 — Robustness Experiments: RUL Target, Operating Regimes & Validation

**Purpose:** improve the pooled XGBoost RUL model based on validation evidence—not by chasing the official test labels.

This notebook runs a structured ablation across:

1. **RUL target cap:** 125, 175, 200, or uncapped.
2. **Feature strategy:** current rolling-feature baseline versus operating-regime-aware sensor normalization.
3. **Multi-horizon validation:** pseudo-truncated engine observations with true RUL targets of 15, 30, 60, 90, 120, 150, 180, 220 and 260 cycles whenever that cutoff exists in an engine trajectory.
4. **A locked audit split:** the selected configuration is evaluated on engines not used for configuration selection, with a predeclared five-seed stability check; no seed is chosen using audit performance.
5. **Final refit:** the chosen configuration is then refit on all 709 labelled training engines and saved for the dashboard.

## Why these experiments?

- NASA describes FD001–FD004 as different combinations of operating conditions and fault modes. FD002 and FD004 include six operating conditions, making operating-condition handling a sensible experiment. [NASA PCoE dataset repository](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/) · [NASA dataset overview](https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data).
- Public C-MAPSS implementations have reported that **125-cycle target clipping can materially change high-RUL error and the frequency of late predictions**. We will compare caps while holding other pieces fixed during the first ablation. [Target-cap ablation repository](https://github.com/luisesparza3347/nasa-cmapss-rul).
- Another public C-MAPSS benchmark reports useful gains from operating-regime clustering plus regime-wise sensor normalization on FD002/FD004, while also warning that RMSE and the asymmetric NASA score can rank models differently. These numbers are not directly comparable to ours because features and protocols differ; the idea is worth testing locally. [Regime-aware C-MAPSS benchmark](https://github.com/eastani/predictive-maintenance-cmapss).
- Repeated-seed reporting is a useful robustness check; a single run should not be presented as proof of stable performance. [C-MAPSS reproducibility / multi-seed example](https://github.com/Orthon-io/cmapss).

## Important honesty note

The project’s previous Notebook 03 already evaluated the official test labels for the original model. Therefore, the official test set has been *seen during project development* and should not be described as a pristine, never-inspected blind holdout. This notebook keeps all new model selection inside the labelled training trajectories and introduces a fresh engine-level validation/audit design. After this notebook chooses the model, run Notebook 03 once to report the new model’s official benchmark comparison, and **do not tune further based on that test result**.

No result is guaranteed to improve. A robust project reports trade-offs, per-subset errors and limitations—not only the most flattering metric.

In [ ]:
from pathlib import Path
import json
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GroupKFold, RandomizedSearchCV, ParameterSampler
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

warnings.filterwarnings("ignore", category=FutureWarning)
RANDOM_STATE = 42
SUBSETS = ["FD001", "FD002", "FD003", "FD004"]
COLUMN_NAMES = ["unit_nr", "time_cycles", "setting_1", "setting_2", "setting_3"] + [f"s_{i}" for i in range(1, 22)]
SENSOR_COLS = [f"s_{i}" for i in range(1, 22)]
SETTING_COLS = ["setting_1", "setting_2", "setting_3"]
ROLLING_WINDOW = 5
RUL_CAPS = [125, 175, 200, None]
VALIDATION_RUL_LEVELS = [15, 30, 60, 90, 120, 150, 180, 220, 260]
N_REGIMES = 6

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
elif not (PROJECT_ROOT / "data").exists() and (PROJECT_ROOT.parent / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
RAW_DIR = PROJECT_ROOT / "data" / "raw"
CMAPSS_DIR = PROJECT_ROOT / "CMAPSSData"
MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
for folder in (RAW_DIR, MODELS_DIR, REPORTS_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def find_dataset_file(filename):
    for candidate in (RAW_DIR / filename, CMAPSS_DIR / filename, PROJECT_ROOT / filename):
        if candidate.exists():
            return candidate
    return None


def load_cmapss_file(filename):
    path = find_dataset_file(filename)
    if path is None:
        raise FileNotFoundError(f"Missing {filename}. Expected it in data/raw/, CMAPSSData/ or project root.")
    frame = pd.read_csv(path, sep=r"\s+", header=None, names=COLUMN_NAMES, engine="python").dropna(axis=1, how="all")
    if frame.shape[1] != len(COLUMN_NAMES):
        raise ValueError(f"Unexpected number of columns in {path}: {frame.shape[1]}")
    return frame


def load_training_subsets():
    parts = []
    missing = []
    for subset in SUBSETS:
        filename = f"train_{subset}.txt"
        if find_dataset_file(filename) is None:
            missing.append(filename)
    if missing:
        raise FileNotFoundError("Missing training data: " + ", ".join(missing))
    for subset in SUBSETS:
        frame = load_cmapss_file(f"train_{subset}.txt")
        frame["subset"] = subset
        frame["engine_key"] = subset + "_" + frame["unit_nr"].astype(int).astype(str).str.zfill(3)
        parts.append(frame)
    return pd.concat(parts, ignore_index=True)

train_raw = load_training_subsets()
print("Training files found for FD001–FD004.")
print(f"Rows: {len(train_raw):,} | engines: {train_raw['engine_key'].nunique():,}")
display(train_raw.groupby("subset").agg(rows=("time_cycles", "size"), engines=("engine_key", "nunique")))

## 1. Build causal rolling features and an engine-wise development/validation/audit split

All rolling features use only the current and preceding cycles within that engine. `engine_key` is a grouping identifier and is never a predictor. A fresh split is made **by whole engine inside every subset**: 60% development, 20% configuration validation, and 20% locked audit. This prevents cycles from the same engine being distributed across partitions.

In [ ]:
def engineer_base_features(frame, window=ROLLING_WINDOW):
    ordered = frame.sort_values(["subset", "unit_nr", "time_cycles"]).reset_index(drop=True).copy()
    required = ["subset", "unit_nr", "engine_key", "time_cycles"] + SETTING_COLS + SENSOR_COLS
    absent = [c for c in required if c not in ordered.columns]
    if absent:
        raise ValueError(f"Missing raw columns: {absent}")
    features = ordered[required].copy()
    groups = ordered.groupby(["subset", "unit_nr"], sort=False)
    for sensor in SENSOR_COLS:
        signal = groups[sensor]
        features[f"{sensor}_roll_mean_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=1).mean())
        features[f"{sensor}_roll_std_{window}"] = signal.transform(lambda s: s.rolling(window=window, min_periods=2).std(ddof=0)).fillna(0.0)
        features[f"{sensor}_delta_1"] = signal.diff().fillna(0.0)
    for subset in SUBSETS:
        features[f"subset_{subset}"] = (ordered["subset"] == subset).astype(np.int8)
    # The target is created from training run-to-failure trajectories only.
    end_cycle = ordered.groupby("engine_key")["time_cycles"].max()
    features["RUL_true"] = end_cycle.reindex(ordered["engine_key"]).to_numpy() - ordered["time_cycles"].to_numpy()
    return features

base = engineer_base_features(train_raw)
ID_COLS = {"subset", "unit_nr", "engine_key", "RUL_true"}
BASE_FEATURE_COLUMNS = [c for c in base.columns if c not in ID_COLS]
assert "engine_key" not in BASE_FEATURE_COLUMNS and "unit_nr" not in BASE_FEATURE_COLUMNS
assert base[BASE_FEATURE_COLUMNS].notna().all().all()
print("Baseline feature matrix:", base[BASE_FEATURE_COLUMNS].shape)

DEV_KEYS, VAL_KEYS, AUDIT_KEYS = [], [], []
for subset in SUBSETS:
    keys = np.sort(base.loc[base["subset"] == subset, "engine_key"].unique())
    dev_keys, remainder = train_test_split(keys, test_size=0.40, random_state=RANDOM_STATE)
    val_keys, audit_keys = train_test_split(remainder, test_size=0.50, random_state=RANDOM_STATE + 1)
    DEV_KEYS.extend(dev_keys.tolist())
    VAL_KEYS.extend(val_keys.tolist())
    AUDIT_KEYS.extend(audit_keys.tolist())

split_rows = []
for subset in SUBSETS:
    split_rows.append({
        "subset": subset,
        "development_engines": int(base.loc[base["subset"].eq(subset) & base["engine_key"].isin(DEV_KEYS), "engine_key"].nunique()),
        "validation_engines": int(base.loc[base["subset"].eq(subset) & base["engine_key"].isin(VAL_KEYS), "engine_key"].nunique()),
        "locked_audit_engines": int(base.loc[base["subset"].eq(subset) & base["engine_key"].isin(AUDIT_KEYS), "engine_key"].nunique()),
    })
display(pd.DataFrame(split_rows))
print(f"Engine split: development={len(DEV_KEYS)}, validation={len(VAL_KEYS)}, audit={len(AUDIT_KEYS)}")

is_dev = base["engine_key"].isin(DEV_KEYS)
is_val = base["engine_key"].isin(VAL_KEYS)
is_audit = base["engine_key"].isin(AUDIT_KEYS)
assert set(DEV_KEYS).isdisjoint(VAL_KEYS) and set(DEV_KEYS).isdisjoint(AUDIT_KEYS) and set(VAL_KEYS).isdisjoint(AUDIT_KEYS)


def capped_target(values, cap):
    values = np.asarray(values, dtype=float)
    return values if cap is None else np.minimum(values, float(cap))


def build_pseudo_points(features, engine_keys, levels=VALIDATION_RUL_LEVELS):
    """Sample realistic pre-failure cutoffs; only use a horizon if that engine can support it."""
    subset = features.loc[features["engine_key"].isin(set(engine_keys))]
    records = []
    for engine_key, group in subset.groupby("engine_key", sort=True):
        final_cycle = int(group["time_cycles"].max())
        for horizon in levels:
            cutoff = final_cycle - int(horizon)
            if cutoff < 1:
                continue
            matches = group.loc[group["time_cycles"].eq(cutoff)]
            if matches.empty:
                continue
            row = matches.iloc[0]
            records.append({
                "base_row_idx": int(matches.index[0]),
                "subset": str(row["subset"]),
                "unit_nr": int(row["unit_nr"]),
                "engine_key": str(engine_key),
                "cutoff_cycle": int(cutoff),
                "true_rul_cycles": float(horizon),
                "validation_horizon_cycles": int(horizon),
            })
    return pd.DataFrame(records)

val_points = build_pseudo_points(base, VAL_KEYS)
audit_points = build_pseudo_points(base, AUDIT_KEYS)
print(f"Validation pseudo-cutoff cases: {len(val_points):,} across {val_points['engine_key'].nunique()} engines")
print(f"Audit pseudo-cutoff cases: {len(audit_points):,} across {audit_points['engine_key'].nunique()} engines")
display(val_points.groupby("validation_horizon_cycles").agg(cases=("engine_key", "size"), engines=("engine_key", "nunique")).reset_index())

## 2. Tune a baseline only on development engines

A small grouped-CV search is run using the conventional cap of 125 and baseline features. The selected parameters are then held fixed across the first eight target/feature ablations, so those runs primarily isolate the target-cap and feature-strategy effects. The winning configuration is retuned later, still without using the locked audit engines.

In [ ]:
PARAM_DISTRIBUTIONS = {
    "n_estimators": [350, 450, 550],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.80, 0.90, 1.00],
    "colsample_bytree": [0.80, 0.90, 1.00],
    "min_child_weight": [3, 5],
    "reg_lambda": [3.0, 7.0],
}
N_BASE_SEARCH = 5  # deliberately compact for a laptop; increase only if time allows
base_dev = base.loc[is_dev].copy()
X_dev = base_dev[BASE_FEATURE_COLUMNS]
y_dev = capped_target(base_dev["RUL_true"], 125)
g_dev = base_dev["engine_key"].astype(str)
base_search = RandomizedSearchCV(
    estimator=XGBRegressor(objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE, n_jobs=4),
    param_distributions=PARAM_DISTRIBUTIONS,
    n_iter=N_BASE_SEARCH,
    scoring="neg_mean_absolute_error",
    cv=GroupKFold(n_splits=3),
    n_jobs=1,
    random_state=RANDOM_STATE,
    verbose=1,
    refit=True,
    error_score="raise",
)
base_search.fit(X_dev, y_dev, groups=g_dev)
BASELINE_BEST_PARAMS = dict(base_search.best_params_)
print("Development-only grouped-CV MAE (capped target):", round(-base_search.best_score_, 3), "cycles")
print("Baseline hyperparameters used for the ablation screen:")
print(json.dumps(BASELINE_BEST_PARAMS, indent=2))
pd.DataFrame(base_search.cv_results_).sort_values("rank_test_score").to_csv(REPORTS_DIR / "robustness_baseline_group_cv.csv", index=False)

## 3. Regime-aware features, fitted only on allowed training engines

The regime-aware variant clusters the three operating settings into six groups and adds one-hot operating-regime indicators. It also adds standardized versions of each current sensor and each five-cycle rolling mean, where the mean/standard deviation are learned from **only the engines passed to the fit function**, grouped by subset and operating regime. The raw features are retained; standardized features are added rather than replacing them.

This is an experiment, not a claim that K-means is universally best. The transformer is refit independently inside each CV fold when tuning the finalist, and is fitted on development engines for the initial ablation screen.

In [ ]:
REGIME_SOURCE_COLUMNS = SENSOR_COLS + [f"{s}_roll_mean_{ROLLING_WINDOW}" for s in SENSOR_COLS]
REGIME_FEATURE_COLUMNS = [f"operating_regime_{i}" for i in range(N_REGIMES)] + [f"{c}_regime_z" for c in REGIME_SOURCE_COLUMNS]


def _make_stats(part, source_columns):
    stats = {}
    for col in source_columns:
        vals = pd.to_numeric(part[col], errors="coerce").to_numpy(dtype=float)
        vals = vals[np.isfinite(vals)]
        if len(vals) == 0:
            stats[col] = (0.0, 1.0)
        else:
            sd = float(np.std(vals, ddof=0))
            stats[col] = (float(np.mean(vals)), max(sd, 1e-6))
    return stats


def fit_regime_transformer(features, fit_engine_keys, random_state=RANDOM_STATE, window=None):
    active_window = int(window if window is not None else ROLLING_WINDOW)
    source_columns = SENSOR_COLS + [f"{s}_roll_mean_{active_window}" for s in SENSOR_COLS]
    fit = features.loc[features["engine_key"].isin(set(fit_engine_keys))].copy().reset_index(drop=True)
    if fit.empty:
        raise ValueError("No rows supplied to fit regime transformer.")
    scaler = StandardScaler().fit(fit[SETTING_COLS])
    scaled_settings = scaler.transform(fit[SETTING_COLS])
    km = KMeans(n_clusters=N_REGIMES, n_init=10, random_state=random_state)
    km.fit(scaled_settings)
    fit["_operating_regime"] = km.predict(scaled_settings).astype(int)
    group_stats = {}
    for (subset, regime), group in fit.groupby(["subset", "_operating_regime"], sort=True):
        group_stats[(str(subset), int(regime))] = _make_stats(group, source_columns)
    subset_stats = {}
    for subset, group in fit.groupby("subset", sort=True):
        subset_stats[str(subset)] = _make_stats(group, source_columns)
    global_stats = _make_stats(fit, source_columns)
    return {
        "setting_scaler": scaler,
        "kmeans": km,
        "group_stats": group_stats,
        "subset_stats": subset_stats,
        "global_stats": global_stats,
        "n_regimes": N_REGIMES,
        "source_columns": source_columns,
        "rolling_window": active_window,
        "method": "KMeans on standardized operating settings; sensor/rolling-mean z-scores within subset and regime",
    }


def add_regime_features(features, transformer):
    out = features.copy().reset_index(drop=True)
    scaled_settings = transformer["setting_scaler"].transform(out[SETTING_COLS])
    regime = transformer["kmeans"].predict(scaled_settings).astype(int)
    out["_operating_regime"] = regime
    for r in range(int(transformer["n_regimes"])):
        out[f"operating_regime_{r}"] = (regime == r).astype(np.int8)
    group_stats = transformer["group_stats"]
    subset_stats = transformer["subset_stats"]
    global_stats = transformer["global_stats"]
    group_positions = {}
    for pos, key in enumerate(zip(out["subset"].astype(str), regime)):
        group_positions.setdefault((key[0], int(key[1])), []).append(pos)
    for source_col in transformer["source_columns"]:
        means = np.empty(len(out), dtype=float)
        stds = np.empty(len(out), dtype=float)
        for key, positions in group_positions.items():
            stats = group_stats.get(key)
            if stats is None:
                stats = subset_stats.get(key[0], global_stats)
            mean, sd = stats.get(source_col, global_stats[source_col])
            means[positions] = mean
            stds[positions] = max(float(sd), 1e-6)
        out[f"{source_col}_regime_z"] = (out[source_col].to_numpy(dtype=float) - means) / stds
    out.drop(columns=["_operating_regime"], inplace=True)
    return out

regime_transform_dev = fit_regime_transformer(base, DEV_KEYS)
base_regime_dev = add_regime_features(base, regime_transform_dev)
print("Baseline feature count:", len(BASE_FEATURE_COLUMNS))
print("Regime-aware feature count:", len(BASE_FEATURE_COLUMNS) + len(REGIME_FEATURE_COLUMNS))
print("Regime transformer fitted on development engines only.")
assert all(c in base_regime_dev.columns for c in REGIME_FEATURE_COLUMNS)
assert not base_regime_dev[BASE_FEATURE_COLUMNS + REGIME_FEATURE_COLUMNS].isna().any().any()

## 4. Multi-horizon validation: target-cap × feature ablation

Each candidate is trained on development engines and evaluated on identical pseudo-truncated validation cases. Targets above the cap are clipped only for fitting; predictions are scored against **uncapped true RUL** as the primary operating view, plus a capped-125 comparison for diagnosis. Because RUL horizons differ, the metrics include error by RUL band and percentage of late/early predictions.

**Primary selection rule:** lowest macro-averaged subset NASA asymmetric penalty, where pseudo-cutoff metrics are averaged within each engine, then within each subset, and finally across FD001–FD004 equally. Macro subset MAE is the tie-breaker. This avoids letting the larger subsets or longer-lived engines dominate configuration selection. This gives more weight to late over-prediction than under-prediction, as in the common C-MAPSS scoring convention. It is a benchmark proxy, not a calibrated monetary loss function.

In [ ]:
def nasa_asymmetric_score(y_true, y_pred):
    error = np.asarray(y_pred, dtype=float) - np.asarray(y_true, dtype=float)
    penalty = np.where(
        error < 0,
        np.exp(np.clip(-error / 13.0, 0, 50)) - 1,
        np.exp(np.clip(error / 10.0, 0, 50)) - 1,
    )
    return float(np.sum(penalty))


def metric_pack(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.maximum(0.0, np.asarray(y_pred, dtype=float))
    err = y_pred - y_true
    return {
        "n_cases": int(len(y_true)),
        "nasa_score_total": nasa_asymmetric_score(y_true, y_pred),
        "nasa_score_per_case": nasa_asymmetric_score(y_true, y_pred) / max(len(y_true), 1),
        "mae_uncapped": float(mean_absolute_error(y_true, y_pred)),
        "rmse_uncapped": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "r2_uncapped": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
        "mean_bias_pred_minus_true": float(np.mean(err)),
        "late_predictions_pct": float(np.mean(err > 0) * 100),
        "early_predictions_pct": float(np.mean(err < 0) * 100),
        "mae_capped125_diagnostic": float(mean_absolute_error(np.minimum(y_true, 125.0), np.clip(y_pred, 0.0, 125.0))),
    }


def macro_metric_pack(point_table, y_pred):
    """Report pooled metrics and a macro view that weights each subset equally.

    Pseudo-cutoff points are repeated within engine and long-lived engines can
    contribute more cutoffs. The macro score first averages within each engine,
    then within subset, then gives each of FD001-FD004 equal weight.
    """
    point_table = point_table.reset_index(drop=True).copy()
    y_pred = np.maximum(0.0, np.asarray(y_pred, dtype=float))
    point_table["_pred"] = y_pred
    point_table["_error"] = y_pred - point_table["true_rul_cycles"].to_numpy(dtype=float)
    engine_rows = []
    for (subset, engine_key), group in point_table.groupby(["subset", "engine_key"], sort=False):
        y_t = group["true_rul_cycles"].to_numpy(dtype=float)
        y_p = group["_pred"].to_numpy(dtype=float)
        engine_rows.append({
            "subset": str(subset),
            "engine_key": str(engine_key),
            "engine_mae": float(np.mean(np.abs(y_p - y_t))),
            "engine_nasa_score_per_case": nasa_asymmetric_score(y_t, y_p) / max(len(y_t), 1),
            "engine_late_pct": float(np.mean(y_p > y_t) * 100),
            "engine_bias": float(np.mean(y_p - y_t)),
        })
    engine_metrics = pd.DataFrame(engine_rows)
    subset_metrics = engine_metrics.groupby("subset", sort=True).agg(
        subset_mae=("engine_mae", "mean"),
        subset_nasa_score=("engine_nasa_score_per_case", "mean"),
        subset_late_pct=("engine_late_pct", "mean"),
        subset_bias=("engine_bias", "mean"),
    )
    pooled = metric_pack(point_table["true_rul_cycles"].to_numpy(dtype=float), y_pred)
    pooled.update({
        "macro_engine_mae_uncapped": float(engine_metrics["engine_mae"].mean()),
        "macro_engine_nasa_score_per_case": float(engine_metrics["engine_nasa_score_per_case"].mean()),
        "macro_subset_mae_uncapped": float(subset_metrics["subset_mae"].mean()),
        "macro_subset_nasa_score_per_case": float(subset_metrics["subset_nasa_score"].mean()),
        "macro_subset_late_predictions_pct": float(subset_metrics["subset_late_pct"].mean()),
        "macro_subset_mean_bias": float(subset_metrics["subset_bias"].mean()),
        "subset_metric_summary_json": json.dumps({
            str(subset): {k: float(v) for k, v in row.items()}
            for subset, row in subset_metrics.to_dict(orient="index").items()
        }, sort_keys=True),
    })
    return pooled


def predict_pseudo_points(model, feature_frame, feature_columns, point_table):
    idx = point_table["base_row_idx"].to_numpy(dtype=int)
    X = feature_frame.iloc[idx][feature_columns]
    return np.maximum(0.0, model.predict(X))

val_points = build_pseudo_points(base, VAL_KEYS)
val_y = val_points["true_rul_cycles"].to_numpy(dtype=float)
mode_frames = {
    "baseline": base,
    "regime_aware": base_regime_dev,
}
mode_columns = {
    "baseline": BASE_FEATURE_COLUMNS,
    "regime_aware": BASE_FEATURE_COLUMNS + REGIME_FEATURE_COLUMNS,
}
config_rows = []
val_predictions_by_config = {}
DEV_ROW_MASK = base["engine_key"].isin(DEV_KEYS)
for mode in ["baseline", "regime_aware"]:
    feat_frame = mode_frames[mode]
    feat_cols = mode_columns[mode]
    X_fit = feat_frame.loc[DEV_ROW_MASK, feat_cols]
    raw_y_fit = base.loc[DEV_ROW_MASK, "RUL_true"].to_numpy(dtype=float)
    for cap in RUL_CAPS:
        y_fit = capped_target(raw_y_fit, cap)
        model = XGBRegressor(
            **BASELINE_BEST_PARAMS,
            objective="reg:squarederror",
            tree_method="hist",
            random_state=RANDOM_STATE,
            n_jobs=4,
        )
        model.fit(X_fit, y_fit)
        pred = predict_pseudo_points(model, feat_frame, feat_cols, val_points)
        metrics = macro_metric_pack(val_points, pred)
        config_name = "uncapped" if cap is None else str(cap)
        row = {"feature_mode": mode, "target_cap": config_name, **metrics}
        config_rows.append(row)
        preds = val_points[["subset", "unit_nr", "engine_key", "cutoff_cycle", "validation_horizon_cycles", "true_rul_cycles"]].copy()
        preds["predicted_rul_cycles"] = pred
        preds["error_pred_minus_true_cycles"] = pred - val_y
        preds["feature_mode"] = mode
        preds["target_cap"] = config_name
        val_predictions_by_config[(mode, config_name)] = preds
        print(f"Finished screen: mode={mode:12s}, cap={config_name:8s}, macro-subset MAE={metrics['macro_subset_mae_uncapped']:.2f}, macro-subset NASA={metrics['macro_subset_nasa_score_per_case']:.3f}")

config_results = pd.DataFrame(config_rows).sort_values(["macro_subset_nasa_score_per_case", "macro_subset_mae_uncapped"]).reset_index(drop=True)
config_results.to_csv(REPORTS_DIR / "robustness_cap_regime_ablation.csv", index=False)
display(config_results[["feature_mode", "target_cap", "n_cases", "macro_subset_nasa_score_per_case", "macro_subset_mae_uncapped", "nasa_score_per_case", "mae_uncapped", "rmse_uncapped", "r2_uncapped", "late_predictions_pct", "early_predictions_pct", "mae_capped125_diagnostic"]].round(3))

winner_row = config_results.iloc[0]
WINNER_MODE = str(winner_row["feature_mode"])
WINNER_CAP_LABEL = str(winner_row["target_cap"])
WINNER_CAP = None if WINNER_CAP_LABEL == "uncapped" else int(WINNER_CAP_LABEL)
print("Validation-selected configuration:")
print(json.dumps({"feature_mode": WINNER_MODE, "target_cap": WINNER_CAP_LABEL, "selection_metric": "lowest macro-averaged subset NASA asymmetric penalty (engine-macro within subset); macro subset MAE tie-breaker"}, indent=2))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.8))
for mode in ["baseline", "regime_aware"]:
    part = config_results.loc[config_results["feature_mode"] == mode].sort_values("target_cap", key=lambda s: s.map({"125":0,"175":1,"200":2,"uncapped":3}))
    axes[0].plot(part["target_cap"], part["mae_uncapped"], marker="o", label=mode)
    axes[1].plot(part["target_cap"], part["macro_subset_nasa_score_per_case"], marker="o", label=mode)
axes[0].set(title="Validation MAE against raw RUL", xlabel="Training target cap", ylabel="MAE (cycles)")
axes[1].set(title="Validation mean asymmetric penalty", xlabel="Training target cap", ylabel="Macro subset NASA penalty / engine")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "robustness_cap_regime_ablation.png", dpi=150, bbox_inches="tight")
plt.show()

winner_val_preds = val_predictions_by_config[(WINNER_MODE, WINNER_CAP_LABEL)].copy()
winner_val_preds.to_csv(REPORTS_DIR / "robustness_winner_validation_predictions.csv", index=False)
winner_val_preds["rul_band"] = pd.cut(winner_val_preds["true_rul_cycles"], bins=[0, 30, 60, 125, 180, np.inf], labels=["1-30", "31-60", "61-125", "126-180", "181+"], include_lowest=True)
val_band_summary = winner_val_preds.groupby(["subset", "rul_band"], observed=False).agg(
    cases=("true_rul_cycles", "size"),
    mae=("error_pred_minus_true_cycles", lambda x: float(np.mean(np.abs(x)))),
    bias=("error_pred_minus_true_cycles", "mean"),
    late_pct=("error_pred_minus_true_cycles", lambda x: float(np.mean(x > 0) * 100)),
).reset_index()
val_band_summary.to_csv(REPORTS_DIR / "robustness_winner_validation_by_subset_rul_band.csv", index=False)
display(val_band_summary.round(2))

## 5. Rolling-window sensitivity for the selected cap/feature strategy

The initial screen varies target cap and regime-aware features at a five-cycle window. Now hold the selected cap and feature strategy fixed, and compare 5-, 10- and 20-cycle rolling windows on the same validation engines. The audit engines remain untouched. The hyperparameters are held fixed for this quick window ablation; the selected candidate is retuned in the next section.

In [ ]:
WINDOW_CANDIDATES = [5, 10, 20]
window_base_frames = {ROLLING_WINDOW: base}
window_screen_predictions = {}
window_rows = []

for window_candidate in WINDOW_CANDIDATES:
    candidate_base = base if window_candidate == ROLLING_WINDOW else engineer_base_features(train_raw, window=window_candidate)
    candidate_base_columns = [c for c in candidate_base.columns if c not in ID_COLS]
    dev_mask_candidate = candidate_base["engine_key"].isin(DEV_KEYS)
    if WINNER_MODE == "regime_aware":
        candidate_transformer = fit_regime_transformer(candidate_base, DEV_KEYS, window=window_candidate)
        candidate_features = add_regime_features(candidate_base, candidate_transformer)
        candidate_regime_cols = [f"operating_regime_{i}" for i in range(N_REGIMES)] + [f"{c}_regime_z" for c in (SENSOR_COLS + [f"{s}_roll_mean_{window_candidate}" for s in SENSOR_COLS])]
        candidate_columns = candidate_base_columns + candidate_regime_cols
    else:
        candidate_transformer = None
        candidate_features = candidate_base
        candidate_columns = candidate_base_columns
    model = XGBRegressor(**BASELINE_BEST_PARAMS, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE, n_jobs=4)
    model.fit(candidate_features.loc[dev_mask_candidate, candidate_columns], capped_target(candidate_base.loc[dev_mask_candidate, "RUL_true"], WINNER_CAP))
    candidate_points = build_pseudo_points(candidate_base, VAL_KEYS)
    candidate_pred = predict_pseudo_points(model, candidate_features, candidate_columns, candidate_points)
    stats = macro_metric_pack(candidate_points, candidate_pred)
    window_rows.append({"rolling_window":window_candidate,"feature_mode":WINNER_MODE,"target_cap":WINNER_CAP_LABEL,**stats})
    points = candidate_points.copy()
    points["predicted_rul_cycles"] = candidate_pred
    points["error_pred_minus_true_cycles"] = candidate_pred - points["true_rul_cycles"]
    window_screen_predictions[window_candidate] = points
    window_base_frames[window_candidate] = candidate_base
    print(f"Window {window_candidate}: macro-subset MAE={stats['macro_subset_mae_uncapped']:.2f}, macro-subset NASA={stats['macro_subset_nasa_score_per_case']:.3f}")

window_results = pd.DataFrame(window_rows).sort_values(["macro_subset_nasa_score_per_case", "macro_subset_mae_uncapped"]).reset_index(drop=True)
window_results.to_csv(REPORTS_DIR / "robustness_window_sensitivity.csv", index=False)
display(window_results[["rolling_window","n_cases","macro_subset_nasa_score_per_case","macro_subset_mae_uncapped","nasa_score_per_case","mae_uncapped","rmse_uncapped","r2_uncapped","late_predictions_pct","early_predictions_pct"]].round(3))
WINNER_WINDOW = int(window_results.iloc[0]["rolling_window"])
print(f"Selected rolling window: {WINNER_WINDOW} cycles")

# Switch the rest of the notebook to the chosen feature-generation window and rebuild the feature-column contracts.
ROLLING_WINDOW = WINNER_WINDOW
base = window_base_frames[WINNER_WINDOW]
BASE_FEATURE_COLUMNS = [c for c in base.columns if c not in ID_COLS]
REGIME_SOURCE_COLUMNS = SENSOR_COLS + [f"{s}_roll_mean_{ROLLING_WINDOW}" for s in SENSOR_COLS]
REGIME_FEATURE_COLUMNS = [f"operating_regime_{i}" for i in range(N_REGIMES)] + [f"{c}_regime_z" for c in REGIME_SOURCE_COLUMNS]
is_dev = base["engine_key"].isin(DEV_KEYS)
is_val = base["engine_key"].isin(VAL_KEYS)
is_audit = base["engine_key"].isin(AUDIT_KEYS)
val_points = build_pseudo_points(base, VAL_KEYS)
audit_points = build_pseudo_points(base, AUDIT_KEYS)
selected_validation_predictions = window_screen_predictions[WINNER_WINDOW].copy()
selected_validation_predictions.to_csv(REPORTS_DIR / "robustness_selected_validation_predictions.csv", index=False)
selected_validation_predictions["rul_band"] = pd.cut(selected_validation_predictions["true_rul_cycles"], bins=[0,30,60,125,180,np.inf], labels=["1-30","31-60","61-125","126-180","181+"], include_lowest=True)
selected_validation_band = selected_validation_predictions.groupby(["subset","rul_band"], observed=False).agg(
    cases=("engine_key","size"),
    mae=("error_pred_minus_true_cycles",lambda x:float(np.mean(np.abs(x)))),
    bias=("error_pred_minus_true_cycles","mean"),
    late_pct=("error_pred_minus_true_cycles",lambda x:float(np.mean(x>0)*100)),
).reset_index()
selected_validation_band.to_csv(REPORTS_DIR / "robustness_selected_validation_by_subset_rul_band.csv",index=False)
display(selected_validation_band.round(2))

## 6. Retune the selected configuration using grouped, multi-horizon CV

Now that the target cap and feature mode have been chosen from validation, tune only that configuration. The CV split is by engine. For regime-aware models, the scaler, K-means clusterer and sensor normalization statistics are refit **inside each fold on that fold’s training engines only**. Each fold’s score is calculated at pseudo-truncated horizons, so the tuning objective aligns with the RUL use case rather than counting every cycle row equally.

The locked audit engines are still unused at this point.

In [ ]:
# Candidate hyperparameters for the selected configuration. Compact search to keep it laptop-friendly.
WINNER_PARAM_DISTRIBUTIONS = {
    "n_estimators": [350, 450, 550],
    "max_depth": [3, 4, 5],
    "learning_rate": [0.03, 0.05],
    "subsample": [0.80, 0.90, 1.00],
    "colsample_bytree": [0.80, 0.90, 1.00],
    "min_child_weight": [3, 5],
    "reg_lambda": [3.0, 7.0],
}
N_WINNER_CANDIDATES = 4
TUNING_KEYS = sorted(set(DEV_KEYS + VAL_KEYS))
param_candidates = list(ParameterSampler(WINNER_PARAM_DISTRIBUTIONS, n_iter=N_WINNER_CANDIDATES, random_state=RANDOM_STATE + 22))
# Always include the development baseline params so the finalist doesn't lose the known baseline by chance.
param_candidates.insert(0, BASELINE_BEST_PARAMS)
# Keep unique candidate dictionaries in a stable order.
unique_candidates, seen = [], set()
for params in param_candidates:
    sig = tuple(sorted(params.items()))
    if sig not in seen:
        seen.add(sig); unique_candidates.append(dict(params))
param_candidates = unique_candidates[:N_WINNER_CANDIDATES + 1]

gkf = GroupKFold(n_splits=3)
cv_score_rows = []
tuning_rows = base.loc[base["engine_key"].isin(TUNING_KEYS)].copy().reset_index(drop=True)
for candidate_id, params in enumerate(param_candidates, start=1):
    fold_metrics = []
    for fold_id, (idx_train, idx_valid) in enumerate(gkf.split(tuning_rows, groups=tuning_rows["engine_key"]), start=1):
        fold_train_keys = tuning_rows.iloc[idx_train]["engine_key"].unique().tolist()
        fold_valid_keys = tuning_rows.iloc[idx_valid]["engine_key"].unique().tolist()
        train_mask = base["engine_key"].isin(fold_train_keys)
        if WINNER_MODE == "regime_aware":
            transformer = fit_regime_transformer(base, fold_train_keys, random_state=RANDOM_STATE + fold_id)
            fold_features = add_regime_features(base, transformer)
            fold_columns = BASE_FEATURE_COLUMNS + REGIME_FEATURE_COLUMNS
        else:
            transformer = None
            fold_features = base
            fold_columns = BASE_FEATURE_COLUMNS
        train_target = capped_target(base.loc[train_mask, "RUL_true"], WINNER_CAP)
        fitted = XGBRegressor(**params, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE + fold_id, n_jobs=4)
        fitted.fit(fold_features.loc[train_mask, fold_columns], train_target)
        fold_points = build_pseudo_points(base, fold_valid_keys)
        raw_pred = predict_pseudo_points(fitted, fold_features, fold_columns, fold_points)
        score = macro_metric_pack(fold_points, raw_pred)
        fold_metrics.append(score)
    row = {
        "candidate_id": candidate_id,
        "mean_cv_macro_subset_nasa_score": float(np.mean([x["macro_subset_nasa_score_per_case"] for x in fold_metrics])),
        "std_cv_macro_subset_nasa_score": float(np.std([x["macro_subset_nasa_score_per_case"] for x in fold_metrics], ddof=1)),
        "mean_cv_macro_subset_mae": float(np.mean([x["macro_subset_mae_uncapped"] for x in fold_metrics])),
        "mean_cv_pooled_nasa_score_per_case": float(np.mean([x["nasa_score_per_case"] for x in fold_metrics])),
        "mean_cv_raw_mae": float(np.mean([x["mae_uncapped"] for x in fold_metrics])),
        "mean_cv_raw_rmse": float(np.mean([x["rmse_uncapped"] for x in fold_metrics])),
        "params": json.dumps(params, sort_keys=True),
    }
    cv_score_rows.append(row)
    print(f"Winner tune candidate {candidate_id}/{len(param_candidates)}: macro-subset NASA={row['mean_cv_macro_subset_nasa_score']:.3f}, macro-subset MAE={row['mean_cv_macro_subset_mae']:.2f}")

winner_cv_results = pd.DataFrame(cv_score_rows).sort_values(["mean_cv_macro_subset_nasa_score", "mean_cv_macro_subset_mae"]).reset_index(drop=True)
winner_cv_results.to_csv(REPORTS_DIR / "robustness_winner_grouped_cv.csv", index=False)
display(winner_cv_results)
WINNER_BEST_PARAMS = json.loads(winner_cv_results.iloc[0]["params"])
print("Selected configuration's retuned XGBoost parameters:")
print(json.dumps(WINNER_BEST_PARAMS, indent=2))

## 7. Locked audit evaluation (one configuration only)

The audit engine IDs have not participated in configuration selection or hyperparameter tuning. We fit the selected configuration on development+validation engines, then assess its fixed hyperparameters on the audit engines at the same feasible horizons over five predeclared random seeds. Seed 42 is the primary audit run; the other seeds quantify model stochasticity, and none is selected based on audit score. The audit provides the most honest internal estimate available in this project version.

Bootstrap intervals resample **engines**, stratified within each subset, so multiple cutoffs from one engine are not treated as independent sampling units. We also predeclare five XGBoost random seeds for a seed-sensitivity check; no seed is selected based on its audit score.

In [ ]:
FIT_KEYS_FOR_AUDIT = sorted(set(DEV_KEYS + VAL_KEYS))
fit_mask_audit = base["engine_key"].isin(FIT_KEYS_FOR_AUDIT)
if WINNER_MODE == "regime_aware":
    audit_transformer = fit_regime_transformer(base, FIT_KEYS_FOR_AUDIT)
    audit_features = add_regime_features(base, audit_transformer)
    WINNER_FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + REGIME_FEATURE_COLUMNS
else:
    audit_transformer = None
    audit_features = base
    WINNER_FEATURE_COLUMNS = BASE_FEATURE_COLUMNS

audit_model = XGBRegressor(**WINNER_BEST_PARAMS, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE, n_jobs=4)
audit_model.fit(audit_features.loc[fit_mask_audit, WINNER_FEATURE_COLUMNS], capped_target(base.loc[fit_mask_audit, "RUL_true"], WINNER_CAP))
locked_audit_points = build_pseudo_points(base, AUDIT_KEYS)
audit_pred = predict_pseudo_points(audit_model, audit_features, WINNER_FEATURE_COLUMNS, locked_audit_points)
audit_y = locked_audit_points["true_rul_cycles"].to_numpy(dtype=float)
audit_metrics = metric_pack(audit_y, audit_pred)
audit_macro_metrics = macro_metric_pack(locked_audit_points, audit_pred)

locked_audit_predictions = locked_audit_points.copy()
locked_audit_predictions["predicted_rul_cycles"] = audit_pred
locked_audit_predictions["error_pred_minus_true_cycles"] = audit_pred - audit_y
locked_audit_predictions["absolute_error_cycles"] = np.abs(audit_pred - audit_y)
locked_audit_predictions.to_csv(REPORTS_DIR / "robustness_locked_audit_predictions.csv", index=False)

# Pre-declared seed stability analysis: same selected configuration, same fixed fit/audit
# split; seeds are never selected by their audit score. Seed 42 is the primary audit run.
AUDIT_SEEDS = [42, 123, 2024, 31415, 7]
seed_stability_rows = []
for audit_seed in AUDIT_SEEDS:
    if audit_seed == RANDOM_STATE:
        seed_pred = audit_pred
    else:
        seed_model = XGBRegressor(**WINNER_BEST_PARAMS, objective="reg:squarederror", tree_method="hist", random_state=audit_seed, n_jobs=4)
        seed_model.fit(audit_features.loc[fit_mask_audit, WINNER_FEATURE_COLUMNS], capped_target(base.loc[fit_mask_audit, "RUL_true"], WINNER_CAP))
        seed_pred = np.maximum(0.0, seed_model.predict(audit_features.iloc[locked_audit_points["base_row_idx"].to_numpy(dtype=int)][WINNER_FEATURE_COLUMNS]))
    seed_metrics = macro_metric_pack(locked_audit_points, seed_pred)
    seed_stability_rows.append({
        "seed": int(audit_seed),
        "macro_subset_nasa_score": float(seed_metrics["macro_subset_nasa_score_per_case"]),
        "macro_subset_mae": float(seed_metrics["macro_subset_mae_uncapped"]),
        "pooled_case_mae": float(seed_metrics["mae_uncapped"]),
        "late_predictions_pct": float(seed_metrics["late_predictions_pct"]),
        "early_predictions_pct": float(seed_metrics["early_predictions_pct"]),
    })
seed_stability = pd.DataFrame(seed_stability_rows)
seed_stability.to_csv(REPORTS_DIR / "robustness_audit_seed_stability.csv", index=False)
display(seed_stability.round(3))

per_engine = locked_audit_predictions.groupby("engine_key").agg(engine_mae=("absolute_error_cycles", "mean"), engine_bias=("error_pred_minus_true_cycles", "mean"), subset=("subset", "first")).reset_index()
# Stratified engine bootstrap gives each subset equal influence, while resampling whole engines.
rng = np.random.default_rng(RANDOM_STATE + 101)
subsets_for_boot = sorted(per_engine["subset"].unique())
boot = np.empty(1500, dtype=float)
for i in range(len(boot)):
    subset_means = []
    for subset_name in subsets_for_boot:
        vals = per_engine.loc[per_engine["subset"] == subset_name, "engine_mae"].to_numpy(dtype=float)
        sample = rng.choice(vals, size=len(vals), replace=True)
        subset_means.append(sample.mean())
    boot[i] = float(np.mean(subset_means))
mae_ci = np.quantile(boot, [0.025, 0.975])

audit_summary = {
    "dataset": "NASA C-MAPSS FD001-FD004 training-engine locked audit",
    "selection": {"feature_mode": WINNER_MODE, "target_cap": WINNER_CAP_LABEL, "best_params": WINNER_BEST_PARAMS},
    "audit_engines": int(per_engine["engine_key"].nunique()),
    "pseudo_cutoff_cases": int(len(locked_audit_predictions)),
    "metrics_raw_rul_pooled_cases": audit_metrics,
    "metrics_raw_rul_macro_subset": {k: audit_macro_metrics[k] for k in ["macro_subset_nasa_score_per_case", "macro_subset_mae_uncapped", "macro_subset_late_predictions_pct", "macro_subset_mean_bias", "subset_metric_summary_json"]},
    "engine_bootstrap_95pct_interval_macro_subset_mean_engine_mae": [float(mae_ci[0]), float(mae_ci[1])],
    "seed_stability_summary": {col: {"mean": float(seed_stability[col].mean()), "std": float(seed_stability[col].std(ddof=1))} for col in ["macro_subset_nasa_score", "macro_subset_mae", "pooled_case_mae", "late_predictions_pct"]},
    "note": "Audit engines were not used for configuration selection or hyperparameter tuning. Five predeclared model seeds quantify model stochasticity on the same locked audit. Metrics are internal pseudo-truncation results, not official NASA test-set metrics.",
}
(REPORTS_DIR / "robustness_locked_audit_summary.json").write_text(json.dumps(audit_summary, indent=2))
print(json.dumps(audit_summary, indent=2))

locked_audit_predictions["rul_band"] = pd.cut(locked_audit_predictions["true_rul_cycles"], bins=[0, 30, 60, 125, 180, np.inf], labels=["1-30", "31-60", "61-125", "126-180", "181+"], include_lowest=True)
audit_by_subset_band = locked_audit_predictions.groupby(["subset", "rul_band"], observed=False).agg(
    cases=("engine_key", "size"),
    mae=("absolute_error_cycles", "mean"),
    rmse=("error_pred_minus_true_cycles", lambda x: float(np.sqrt(np.mean(np.square(x))))),
    bias=("error_pred_minus_true_cycles", "mean"),
    late_pct=("error_pred_minus_true_cycles", lambda x: float(np.mean(x > 0) * 100)),
).reset_index()
audit_by_subset_band.to_csv(REPORTS_DIR / "robustness_locked_audit_by_subset_rul_band.csv", index=False)
display(audit_by_subset_band.round(2))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].scatter(audit_y, audit_pred, alpha=0.65)
upper = float(max(np.max(audit_y), np.max(audit_pred), 1.0))
axes[0].plot([0, upper], [0, upper], "--", color="gray")
axes[0].set(title="Locked audit: predicted vs true RUL", xlabel="True RUL (cycles)", ylabel="Predicted RUL (cycles)")
axes[1].hist(audit_pred - audit_y, bins=25, edgecolor="white")
axes[1].axvline(0, linestyle="--", color="gray")
axes[1].set(title="Locked audit error distribution", xlabel="Predicted − true RUL (cycles)", ylabel="Pseudo-cutoff cases")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "robustness_locked_audit.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Final refit on all labelled training trajectories and save the robust artifact

The locked-audit metrics and five-seed stability summary above were computed before the final refit and remain the internal audit estimate. For application use, the selected target cap, feature strategy and retuned parameters are now fitted on all training trajectories. Official test trajectories and their RUL labels are not read by this notebook.

The artifact includes the regime transformer when needed. The updated `app.py` and Notebook 03 know how to apply it at inference/evaluation time.

In [ ]:
ALL_ENGINE_KEYS = sorted(base["engine_key"].unique().tolist())
if WINNER_MODE == "regime_aware":
    final_transformer = fit_regime_transformer(base, ALL_ENGINE_KEYS)
    final_features = add_regime_features(base, final_transformer)
    FINAL_FEATURE_COLUMNS = BASE_FEATURE_COLUMNS + REGIME_FEATURE_COLUMNS
else:
    final_transformer = None
    final_features = base
    FINAL_FEATURE_COLUMNS = BASE_FEATURE_COLUMNS

final_model = XGBRegressor(**WINNER_BEST_PARAMS, objective="reg:squarederror", tree_method="hist", random_state=RANDOM_STATE, n_jobs=4)
final_model.fit(final_features[FINAL_FEATURE_COLUMNS], capped_target(base["RUL_true"], WINNER_CAP))

artifact = {
    "model": final_model,
    "sensor_cols": SENSOR_COLS,
    "setting_cols": SETTING_COLS,
    "feature_columns": FINAL_FEATURE_COLUMNS,
    "base_feature_columns": BASE_FEATURE_COLUMNS,
    "rolling_window": ROLLING_WINDOW,
    "rul_cap": WINNER_CAP,
    "best_params": WINNER_BEST_PARAMS,
    "dataset": "NASA PCoE C-MAPSS FD001-FD004 pooled XGBoost model (robustness-selected)",
    "subsets": SUBSETS,
    "target_unit": "operating cycles",
    "random_state": RANDOM_STATE,
    "n_training_rows": int(len(train_raw)),
    "n_training_engines": int(train_raw["engine_key"].nunique()),
    "n_model_features": int(len(FINAL_FEATURE_COLUMNS)),
    "feature_mode": WINNER_MODE,
    "regime_transformer": final_transformer,
    "experiment_selection": {
        "target_cap": WINNER_CAP_LABEL,
        "feature_mode": WINNER_MODE,
        "selection_metric": "lowest macro-averaged subset NASA asymmetric penalty (engine-macro within subset); macro subset MAE tie-breaker",
        "locked_audit_summary": audit_summary,
        "official_test_used_for_selection": False,
    },
    "target_policy": "RUL_true = final_cycle - current_cycle; target cap selected by training-engine validation; official test labels excluded from fitting and selection",
}
MODEL_PATH = MODELS_DIR / "xgb_rul_model_all_subsets.joblib"
joblib.dump(artifact, MODEL_PATH)

final_summary = {
    "dataset": artifact["dataset"],
    "training_rows": artifact["n_training_rows"],
    "training_engines": artifact["n_training_engines"],
    "features": len(FINAL_FEATURE_COLUMNS),
    "selected_feature_mode": WINNER_MODE,
    "selected_training_target_cap": WINNER_CAP_LABEL,
    "best_params": WINNER_BEST_PARAMS,
    "validation_ablation_csv": str(REPORTS_DIR / "robustness_cap_regime_ablation.csv"),
    "locked_audit_summary": audit_summary,
    "model_artifact": str(MODEL_PATH),
    "official_test_used_for_fit_or_selection": False,
}
(REPORTS_DIR / "robustness_final_training_summary.json").write_text(json.dumps(final_summary, indent=2))
print("Final robust model saved:", MODEL_PATH)
print("Training engines / rows:", artifact["n_training_engines"], "/", f"{artifact['n_training_rows']:,}")
print("Selected feature mode / target cap:", WINNER_MODE, "/", WINNER_CAP_LABEL)
print("Model feature count:", len(FINAL_FEATURE_COLUMNS))
print("Next: run notebooks/03_test_evaluation_and_explainability.ipynb ONCE for benchmark reporting, then run streamlit run app.py.")

## 9. Interview-ready interpretation checklist

After the notebook completes, use the actual generated `robustness_cap_regime_ablation.csv` and `robustness_locked_audit_summary.json`—do not copy numbers from other repositories.

Explain:

- **Why four subsets?** They differ in operating conditions and fault-mode combinations.
- **Why target-cap ablation?** A 125-cycle cap is common, but high-RUL labels above 125 then become impossible to learn exactly; compare the trade-off rather than assuming one cap is universally best.
- **Why regime features?** The same sensor value can mean something different under different operating conditions. Added standardized features attempt to separate operating regime from degradation signal.
- **Why engine-level splits?** Consecutive cycles of the same engine are correlated; random row splits would leak engine-specific information.
- **Why multi-horizon pseudo-cutoffs?** The prior validation sampled only up to 125 cycles, so it could not assess the long-horizon errors seen in the uncapped official test metrics.
- **Why a locked audit set?** It gives a more honest internal estimate after the validation set has been used to select the configuration.
- **Why not claim “industry-grade” deployment?** NASA C-MAPSS is simulated data. A real deployment needs external validation on actual machinery, monitoring, uncertainty/coverage assessment, governance and domain approval.

### Public references consulted

- [NASA PCoE C-MAPSS description](https://www.nasa.gov/intelligent-systems-division/discovery-and-systems-health/pcoe/pcoe-data-set-repository/)
- [NASA C-MAPSS data / subset definitions](https://data.nasa.gov/dataset/cmapss-jet-engine-simulated-data)
- [Target-cap ablation example](https://github.com/luisesparza3347/nasa-cmapss-rul)
- [Operating-regime features and cross-subset diagnostic example](https://github.com/eastani/predictive-maintenance-cmapss)
- [Repeated-seed reproducibility example](https://github.com/Orthon-io/cmapss)
